In [2]:
import numpy as np
import torch
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from typing import Any

from project_root import PROJECT_ROOT

import fiftyone as fo
import fiftyone.utils.torch as fout

import torchvision

In [3]:
coco_root = Path("/home/dherrera/data/coco")
data_root = Path("/home/dherrera/data/elephants")

In [4]:
from scripts.model_serialization import load_model


# model_base = load_model(
#     "/home/dherrera/Downloads/train_output/model_base.pth", weights_only=True
# )
model_trained = load_model(
    Path(
        "/home/dherrera/git/zoo_vision/models/model_coco_4cam_8k/maskrcnn_c2_coco_4cam_8k.pth"
    ),
)
model_trained

Loading empty model...
Loading weights from disk...
Restoring weights...


MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
         

In [5]:
with (PROJECT_ROOT / "data/config.json").open() as f:
    config = json.load(f)
classes = [(data["id"], name) for name, data in config["individuals"].items()]
classes = sorted(classes, key=lambda x: x[0])
classes = [f"{x[0]:02}_{x[1]}" for x in classes]
print(classes)
print(fo.list_datasets())

['01_Chandra', '02_Indi', '03_Fahra', '04_Panang', '05_Thai']
['coco-elephants-train2017', 'coco-elephants-val2017', 'zoo-elephants-detection-train', 'zoo-elephants-identity', 'zoo-elephants-identity-train', 'zoo-elephants-identity-val']


In [6]:
# from PIL import Image
# for sample in ds_zoo_elephants.iter_samples():
#   im = np.asarray(Image.open(sample.filepath).convert("RGB"))
#   result = fo_model.predict(im)
#   print(result)
#   sample.add_labels(result, label_field="zoo_identity")
#   break

In [7]:
ds_zoo_elephants = fo.load_dataset("zoo-elephants-detection-train")
session = fo.launch_app(ds_zoo_elephants, auto=False)

weights = torchvision.models.get_weight("MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1")
transforms = weights.transforms()

config = fout.TorchImageModelConfig(
    {
        "entrypoint_fcn": lambda: model_trained,
        "entrypoint_args": {},
        "output_processor_cls": "fiftyone.utils.torch.InstanceSegmenterOutputProcessor",
        "classes": [f"Class {i}" for i in range(91)],
        "transforms": transforms,
        "image_min_dim": 224,
        "image_max_dim": 2048,
    }
)
fo_model = fout.TorchImageModel(config)

predictions_view = ds_zoo_elephants.take(100, seed=51)
predictions_view.apply_model(fo_model, label_field="maskrcnn")
# high_conf_view = predictions_view.filter_labels("zoo maskrcnn", fo.ViewField("confidence") > 0.85, only_matches=False)

session.view = predictions_view
session.open_tab()

Session launched. Run `session.show()` to open the App in a cell output.
 100% |█████████████████| 100/100 [6.0s elapsed, 0s remaining, 20.1 samples/s]      


<IPython.core.display.Javascript object>